    # Trabajo final · Data Engineer · UniAndina

    **Universidad** · Empresa ficticia. Los datos son sintéticos y ningún resultado representa una situación real de ARCA. No hay credenciales ni infraestructura real.

    **Lo propio de la empresa** (la base operacional y los eventos de sus sistemas) **se genera por código en este notebook**. **Lo externo** (los datos que mandan terceros) **llega como archivos CSV y XLSX** en la carpeta `datos_externos/`, junto a este notebook.

    ## El caso

    UniAndina tiene 2.000 alumnos, 40 materias y tres sistemas: la base académica, las notas que cargan los docentes y la biblioteca. Quiere un catálogo de datos con su linaje, controles de calidad, accesos por rol y detectar a tiempo a quienes están en riesgo de abandonar.

    ## Qué tiene que resolver la plataforma

    - Un catálogo con el linaje de al menos un indicador.
- Reglas de calidad medibles.
- Una detección temprana del riesgo de abandono.

    ## Restricciones del caso

    - Hay personas con dos legajos y cambios de plan de estudio.
- Los esquemas de notas cambian entre cuatrimestres.
- Datos personales de estudiantes.
- Los archivos externos llegan en CSV y XLSX, y cambian de formato sin aviso.

    Todo lo que el caso **no declara** es una suposición: escribila y justificala.

    Las instrucciones completas (qué se entrega, rúbrica y defensa) están en el PDF **«Trabajo final · Data Engineer · Instrucciones»**. Este notebook es tu lugar de trabajo: vas del **SQL al análisis** sin salir de acá. Elegí la herramienta **después** de fijar el requisito.

## Paso 0 · Instalación

Corré esta celda primero. Si falta alguna librería (incluida `openpyxl`, para leer XLSX), la instala en el mismo Python que ejecuta este notebook.

In [1]:
# Instalación autocontenida: instala, en el mismo Python que ejecuta este notebook, lo que falte.
import importlib, importlib.util, subprocess, sys

REQUISITOS = {"numpy": "numpy", "pandas": "pandas", "matplotlib": "matplotlib", "openpyxl": "openpyxl"}
print(f"Kernel en uso: {sys.executable} (Python {sys.version.split()[0]})")
faltan = [pip for modulo, pip in REQUISITOS.items() if importlib.util.find_spec(modulo) is None]
if faltan:
    print("Instalando:", ", ".join(faltan), "(puede tardar un minuto)")
    base = [sys.executable, "-m", "pip", "install", "-q"]
    try:
        try:
            subprocess.check_call(base + faltan)
        except subprocess.CalledProcessError:       # el Python del sistema suele rechazar instalaciones
            print("Reintentando con --break-system-packages ...")
            subprocess.check_call(base + ["--break-system-packages"] + faltan)
    except Exception:
        raise SystemExit("No se pudo instalar en este kernel. Elegí el kernel .venv (arriba a la derecha) y volvé a correr esta celda.")
    importlib.invalidate_caches()
    print("Listo.")
else:
    print("Todo instalado: podés seguir.")

Kernel en uso: /usr/local/bin/python3 (Python 3.13.18)
Todo instalado: podés seguir.


## 1 · Las fuentes

La primera celda **genera lo propio** con código comprimido a propósito: no hace falta leerla, y leerla te quita la mitad del trabajo. La segunda lo escribe en la carpeta `fuentes/` (una base SQLite y los eventos en JSON), lista los archivos externos y comprueba que tu base coincide con la que se usó para armarlos.

Te deja estas herramientas: `sql(...)`, `ls()`, `externos()`, `hojas(...)`, `leer_externo(...)`, `leer_eventos(...)`, `dwh()`, `medir(...)`, `regla(...)`, `correr_dag(...)`, `zonas()` y `enmascarar(...)`.

In [2]:
import zlib, base64
_BLOB = (
    "eNqNV+tu2zYU/p+nIAQEkFJHkORLHWcuVrQYNqDNBiwDNgiGQUt0ylYiBUrqWhh+pP0Y+gh5sX2HlHxJnSyGYcskz+G5fOc7"
    "x7KstGmYasvqK+M1U9WAVVzleMS7ys9ysWbLXEnfDJgKZsyIpjWKpXVjfKka/0sQsLU27AuTipkQS+JOmNpPIvcasNG0f1LB"
    "4qxT2FZ+vh4wKMWFtRD5XnWVh5lWGW/8lI7k67DmZVUIe10hFAQDdsGqAOKwVJfLuuGNmFstiwGTd0obsZQqF1/mt6YVgbs0"
    "F5NlqyREpOI+nZ5PJsHsjOFl2Byuh05fiNO8LZqlUXf2XHDNbnCAXLpmGTdGGARnzlLvF3UnlBTm/h/uDZj3Oi+lkggNz+T9"
    "N0VLv9Uy04W+6068FUZkH7R9lLW4/1d7i2vGC2iD3295w38yvBT+xpP5khdtqbQ3Y+nae/1uI2fRON96NtzShpvjev+GnPYK"
    "ccc/dmffbWz02QsmHzuOjOJsn9gbxNJTulwZ0d1mb2ZPX7mGJ3yp4GsphWroch9O3MpSICNl5Xvx1VV0GcV4ewGswWajlw32"
    "c1E03D+AC9AxnIyR1njszEGmmrn31guCEPFck5Dvnf91eV5enuce2dslAreaECGVmfD73EAFgDVPw2Q6YGEywkc83X0kCxKv"
    "Cq6WMLTNpT7UkXpJFE8pQUmUDL3FTtcYloWjsRUWJZeFC5VL0kZuf9yBK2yg91TQtoFFG9CPfPOiB/YkOgVlVEH11Qf2cDw9"
    "gMOCkGcBcUMZPp2hSRQsOskOGZ3Yu83wcWw4qR6NfRXyYkCaTpbWNSt5cxK8WBdGchek97cEpeEJQ0eRxdIh+N47SYu+5HGR"
    "J/OPM1sYZ6RwlZ0VsgLeYH7JCx8YICBEDmphFONjguN8RVlQHDKmIwOkjf1AemzmspZoUDeu/GFFinh9wnNkt7/LuGOXfitr"
    "ac8nYI0uk6zD2PgyzryDk/SS650tqVwwPJHwfL4XsGs7I8nGcDJjSFkjVSuOlNHVpaPnLk4jeE5selCAwwGbBESpAoWRiflP"
    "vKjFA6vo9Ym9mLP42gYBfq8LDYj0ASYK1a3K95F+GSbskg1DKm0EEc4MWBzamxD0OAoQdRQLNHm8MnrFc+6R91b9qzkbMQE7"
    "wFei3u3vJVyI7r9ZEf9hxI6jMxoHTheErymRIa8qAVv9tffmj82nWTQB1gZUlrtas+YCyOEez2mJpawdkBoy3oJhp+q5wiQ1"
    "YGvv7a9vNgdJQEhQgg71QdAj7mF5WRCiqeCSem6ZASs1RQaA2vOE+9GXIRVMyxtjyRmVRiQGsslBDPBC4w4KSUdIyfhJQtIn"
    "6eiK6nV0ol6TMbGKPrK0k3rz52MtxgntvO+5yPquH6Ei9R0T2ew8CNZzw0Oy9J3rDN1NIE42IaWlxx22aCBRQAJBLIrGZERY"
    "6CwtiTWdjoU937eXmHAP+rnEd7RwdVhSL2gBVIuomCirSaHo0KS++i1zLMLc6Mrv/Tq2nXQkT+qwDPK0jtAIZcHWbR8PJPTD"
    "9U5Ooek9nRErF3ItMwwFWnlba0p6vEjh8FUSHi5S44+CjjyiDvm8tF0S/Sc8bNa9E+jNPV6H0ckOmh7hdLHDLzSnrv+T9UZq"
    "Y41yjf9wG0z6Wfd7uLDbc4MPHlfS7T4Ye2yEo+Hzxp44odkniv5/5qHOnHZVu1xjzijs5SDdvz9grLTsu0x7FhzsMQpOIXhi"
    "8BnsSNMWAM8aCQddvFdyZZvaczrZcr/l+kilZV1r5U+RST8G5ScHlB+47hAS8gFxjPrczT+HXHlNBvREeow2TvR+wOavqHc7"
    "MocSN99YtvVkvVIQWHtXL6ebBx0uji4urqjnXFxQ3yGu95r6+6HV1lg0fSx7D7RS8oYRXJ1ORkjiLov16Szenv88O38/O//d"
    "jrCNrGw1VcZerr1uQOz+CW28fIXtTTdjkq2WVTrSot94tNRlWZUWLEF6Dpv1klBMi7zcUrpN9gH5rq1OS43LjlDCrP6MVRV3"
    "lWw3iCX6jYTkxWca8504klVI3YiMhx+ReBqGsbTdnp3hHxFmMOLto39cZ/8Bi3tvuQ=="
)
exec(zlib.decompress(base64.b64decode(_BLOB)).decode("utf-8"))   # define generar(): código comprimido a propósito

In [3]:
import os, json, sqlite3, time, shutil
pd.options.display.width = 200; pd.options.display.max_columns = 40; pd.options.display.float_format = "{:,.3f}".format
t = generar(); DESTINO = "fuentes"; shutil.rmtree(DESTINO, ignore_errors=True); os.makedirs(DESTINO)   # lo propio de la empresa: base operacional y eventos
_con = sqlite3.connect(f"{DESTINO}/operacional.db")
for nombre, df in t["db"].items(): df.astype({c: "object" for c in df.columns if str(df[c].dtype).startswith("str")}).to_sql(nombre, _con, index=False)
_con.close()
if t["eventos"]: os.makedirs(f"{DESTINO}/eventos")
for nombre, ev in t["eventos"].items():
    with open(f"{DESTINO}/eventos/{nombre}", "w", encoding="utf-8") as f:
        for x in ev: f.write(json.dumps(x, ensure_ascii=False) + "\n")
EXTERNOS = next((c for c in ("datos_externos", "../datos_externos") if os.path.isdir(c)), None)
if EXTERNOS is None: print("⚠ No encuentro la carpeta datos_externos. Abrí este notebook desde la carpeta de tu equipo, junto a datos_externos/.")

def sql(consulta, base=f"{DESTINO}/operacional.db", parametros=None):
    """Corre una consulta SQL sobre la base operacional (o sobre otra base) y devuelve un DataFrame."""
    c = sqlite3.connect(base)
    try: return pd.read_sql_query(consulta, c, params=parametros)
    finally: c.close()
def ls(carpeta=DESTINO):
    """Lista lo propio que se generó (base y eventos) con su tamaño."""
    for raiz, _, nombres in sorted(os.walk(carpeta)):
        for n in sorted(nombres): print(f"{os.path.join(raiz, n):<62} {os.path.getsize(os.path.join(raiz, n)) / 1024:>9,.0f} KB")
def externos():
    """Lista los archivos externos (CSV y XLSX) con su tamaño y, en los XLSX, sus hojas."""
    for n in sorted(os.listdir(EXTERNOS)):
        ruta = os.path.join(EXTERNOS, n); extra = ("  hojas: " + ", ".join(pd.ExcelFile(ruta).sheet_names)) if n.endswith(".xlsx") else ""
        print(f"{n:<40} {os.path.getsize(ruta) / 1024:>8,.0f} KB{extra}")
def hojas(nombre):
    """Nombres de las hojas de un archivo XLSX externo."""
    return pd.ExcelFile(os.path.join(EXTERNOS, nombre)).sheet_names
def leer_externo(nombre, hoja=0, **opciones):
    """Lee un archivo externo (CSV o XLSX) como DataFrame. No arregla nada: ver cómo viene es parte del trabajo."""
    ruta = os.path.join(EXTERNOS, nombre)
    return pd.read_excel(ruta, sheet_name=hoja, **opciones) if nombre.endswith(".xlsx") else pd.read_csv(ruta, **opciones)
def leer_eventos(nombre):
    """Lee un archivo de eventos JSON (uno por línea) como DataFrame."""
    return pd.read_json(f"{DESTINO}/eventos/{nombre}", lines=True)
def dwh(archivo="dwh.db"):
    """Conexión al almacén que vas a modelar (un archivo SQLite nuevo)."""
    return sqlite3.connect(archivo)
def medir(funcion, *args, **kwargs):
    """Mide cuánto tarda una función y devuelve (resultado, segundos)."""
    t0 = time.perf_counter(); r = funcion(*args, **kwargs); return r, time.perf_counter() - t0
resultados_calidad = []
def regla(nombre, valor, umbral, mayor_es_mejor=False, descripcion=""):
    """Registra una regla de calidad medible: compara un valor con un umbral y deja el resultado."""
    ok = valor >= umbral if mayor_es_mejor else valor <= umbral
    resultados_calidad.append({"regla": nombre, "valor": valor, "umbral": umbral, "cumple": ok, "descripción": descripcion}); print(("✔" if ok else "✘"), nombre, "→", valor, f"(umbral {umbral})"); return ok
def correr_dag(tareas, dependencias, reintentos=2):
    """Ejecuta tareas (nombre → función) respetando dependencias, con reintentos. Si una falla, sus dependientes no corren."""
    estado = {}; pendientes = list(tareas)
    while pendientes:
        listas = [n for n in pendientes if all(estado.get(d) == "ok" for d in dependencias.get(n, []))]; bloq = [n for n in pendientes if any(estado.get(d) in ("falló", "omitida") for d in dependencias.get(n, []))]
        for n in bloq: estado[n] = "omitida"; pendientes.remove(n); print(f"  ⏭  {n}: omitida (falló una dependencia)")
        if not listas and not bloq: raise RuntimeError(f"Dependencias circulares o faltantes: {pendientes}")
        for n in listas:
            pendientes.remove(n)
            for intento in range(1, reintentos + 2):
                try: tareas[n](); estado[n] = "ok"; print(f"  ✔ {n}" + (f" (intento {intento})" if intento > 1 else "")); break
                except Exception as ex: print(f"  ✘ {n}, intento {intento}: {type(ex).__name__}: {ex}"); estado[n] = "falló"
    return estado
def zonas(base="lake"):
    """Crea las carpetas de las zonas de datos."""
    for z in ("cruda", "curada", "variables", "consumo"): os.makedirs(f"{base}/{z}", exist_ok=True)
    return sorted(os.listdir(base))
def enmascarar(valor, visibles=2):
    """Enmascara un texto dejando visibles los últimos caracteres."""
    s = str(valor); return "*" * max(len(s) - visibles, 0) + s[-visibles:]

HUELLA = {'alumnos': [2060, 0.0], 'materias': [40, 0.0], 'cursadas': [15315, 0.0], 'cambios_plan': [300, 0.0], 'biblioteca.jsonl': 10946}
def comprobar_base():
    """Comprueba que la base que generaste coincide con la que usaron para armar los archivos externos."""
    bien = True
    for nombre, df in t["db"].items():
        filas, suma = len(df), round(float(df.select_dtypes("number").sum().sum()), 2); e_filas, e_suma = HUELLA[nombre]
        if filas != e_filas or abs(suma - e_suma) > max(0.01, abs(e_suma) * 1e-6): bien = False; print(f"⚠ {nombre}: no coincide ({filas} filas)")
    for nombre, ev in t["eventos"].items():
        if len(ev) != HUELLA[nombre]: bien = False; print(f"⚠ {nombre}: no coincide ({len(ev)} eventos)")
    print("✔ Tu base coincide con los archivos externos." if bien else "⚠ Tu base no coincide con los archivos externos: reiniciá el kernel y volvé a correr todo. Si sigue, avisá al facilitador.")
    return bien
print("Lo propio, generado en la carpeta", DESTINO + "/")
ls(); print("\nLo externo, en la carpeta", EXTERNOS + "/")
externos(); print(); comprobar_base()

Lo propio, generado en la carpeta fuentes/
fuentes/operacional.db                                               940 KB
fuentes/eventos/biblioteca.jsonl                                   1,057 KB

Lo externo, en la carpeta datos_externos/
actas_docentes.xlsx                            44 KB  hojas: Ciencias, Humanidades, Gestión
becas.csv                                      10 KB
encuesta_satisfaccion.xlsx                     20 KB  hojas: Respuestas
notas_2024-2c.csv                             187 KB
notas_2025-1c.csv                             179 KB

✔ Tu base coincide con los archivos externos.


True

Los archivos externos **cambian de formato sin aviso**, y los datos propios traen duplicados, datos tardíos y datos personales. Descubrirlo, decidir qué hacer con cada caso y dejarlo escrito es parte de la evidencia. `leer_externo(...)` lee el archivo tal como viene: no arregla nada.

In [4]:
# Explorá lo propio y lo externo
sql("SELECT name FROM sqlite_master WHERE type = 'table'")

,name
0,alumnos
1,materias
2,cursadas
3,cambios_plan


## Descubrimiento de problemas

Antes de diseñar, medimos los problemas de cada fuente. La celda siguiente los cuenta con código: si cambian los datos, la tabla cambia. Abajo está qué hacemos con cada uno y lo que suponemos porque el caso no lo declara.

In [5]:
# Descubrimiento: cada problema se mide con código, para que la evidencia se pueda volver a correr.
import re

def patron(serie):
    """Reemplaza dígitos por 9 para ver los formatos distintos de una columna."""
    return serie.astype(str).str.replace(r"\d", "9", regex=True).value_counts().to_dict()

alumnos  = sql("SELECT * FROM alumnos")
materias = sql("SELECT * FROM materias")
cursadas = sql("SELECT * FROM cursadas")
cambios  = sql("SELECT * FROM cambios_plan")
biblio   = pd.read_json(f"{DESTINO}/eventos/biblioteca.jsonl", lines=True, dtype={"isbn": str, "id_alumno": str})
notas_24 = leer_externo("notas_2024-2c.csv", dtype=str)
notas_25 = leer_externo("notas_2025-1c.csv", dtype=str)
becas    = leer_externo("becas.csv", dtype=str)
actas    = pd.concat([leer_externo("actas_docentes.xlsx", hoja=h, header=3, dtype=str).assign(hoja=h)
                      for h in hojas("actas_docentes.xlsx")], ignore_index=True)
encuesta = leer_externo("encuesta_satisfaccion.xlsx", header=2)

hallazgos = []
def hallazgo(fuente, problema, medido, total):
    hallazgos.append({"fuente": fuente, "problema": problema, "casos": int(medido), "de": int(total),
                      "%": round(100 * medido / total, 1) if total else 0.0})

# --- Lo propio: base académica ---
pares = alumnos[alumnos.dni.duplicated(keep=False)]
hallazgo("alumnos", "Personas con dos legajos (mismo DNI, nombre y correo; legajo L3xxxx)", pares.dni.nunique(), alumnos.dni.nunique())
hallazgo("alumnos", "Pares duplicados con estado_final distinto entre sus dos legajos",
         (pares.groupby("dni").estado_final.nunique() > 1).sum(), pares.dni.nunique())
hallazgo("alumnos", "plan_estudio guarda solo el plan actual (los que cambiaron ya figuran con el nuevo)",
         cambios.merge(alumnos, on="id_alumno").eval("plan_estudio == plan_nuevo").sum(), len(cambios))
hallazgo("cursadas", "Cursadas de alumnos que no existen en la base (id AL9xxxx)", (~cursadas.id_alumno.isin(alumnos.id_alumno)).sum(), len(cursadas))
hallazgo("cursadas", "Legajos duplicados sin ninguna cursada", (~alumnos.id_alumno.isin(cursadas.id_alumno) & alumnos.legajo.str.startswith("L3")).sum(), len(pares) // 2)
x = cursadas.merge(alumnos[["id_alumno", "estado_final"]])
hallazgo("cursadas", "Estado 'abandonó': aparece SOLO en alumnos con estado_final = abandono (es el objetivo escrito de otra forma)",
         ((x.estado == "abandonó") & (x.estado_final == "abandono")).sum(), (x.estado == "abandonó").sum())

# --- Lo propio: eventos de biblioteca ---
hallazgo("biblioteca.jsonl", "Préstamos que identifican al alumno por legajo y no por id_alumno", biblio.id_alumno.str.startswith("L").sum(), len(biblio))
hallazgo("biblioteca.jsonl", "El archivo no viene ordenado por fecha (eventos con fecha anterior al previo)", (pd.to_datetime(biblio.ts).diff().dt.total_seconds() < 0).sum(), len(biblio))

# --- Lo externo: notas por cuatrimestre ---
hallazgo("notas_2025-1c.csv", "Cambio de esquema: id_alumno → id_estudiante y nota → calificacion",
         int(list(notas_25.columns) != list(notas_24.columns)), 1)
hallazgo("notas_2025-1c.csv", "Cambio de escala: notas de 0 a 100 en vez de 0 a 10", (pd.to_numeric(notas_25.calificacion) > 10).sum(), len(notas_25))
n24, n25 = pd.to_numeric(notas_24.nota), pd.to_numeric(notas_25.calificacion)
hallazgo("notas_2024-2c.csv", "Notas fuera de rango (menores a 0 o mayores a 10)", ((n24 < 0) | (n24 > 10)).sum(), len(notas_24))
hallazgo("notas_2025-1c.csv", "Notas fuera de rango (menores a 0 o mayores a 100)", ((n25 < 0) | (n25 > 100)).sum(), len(notas_25))
k = notas_24.astype({"nota": float}).merge(cursadas[cursadas.cuatrimestre == "2024-2c"], on=["id_alumno", "id_materia"])
hallazgo("notas_2024-2c.csv", "Cursadas 'aprobada' con nota válida menor a 4", k.query("estado == 'aprobada' and 0 <= nota < 4").shape[0], len(k))
hallazgo("notas_2024-2c.csv", "Cursadas 'abandonó' que igual tienen nota", (k.estado == "abandonó").sum(), len(k))

# --- Lo externo: actas de los docentes (XLSX) ---
hallazgo("actas_docentes.xlsx", "Encabezado en la fila 4 y cuatrimestre escrito en la celda A2 (no es una columna)", 3, 3)
hallazgo("actas_docentes.xlsx", "Legajo sin la 'L' inicial", (~actas.Legajo.str.upper().str.startswith("L")).sum(), len(actas))
hallazgo("actas_docentes.xlsx", "Legajo con 'l' minúscula", actas.Legajo.str.startswith("l").sum(), len(actas))
hallazgo("actas_docentes.xlsx", "Nota con coma decimal ('7,0')", actas.Nota.str.contains(",").sum(), len(actas))
hallazgo("actas_docentes.xlsx", "Nota 'AUS' (ausente) donde el CSV de notas sí trae un número", (actas.Nota == "AUS").sum(), len(actas))
na = pd.to_numeric(actas.Nota.str.replace(",", "."), errors="coerce")
hallazgo("actas_docentes.xlsx", "Notas fuera de rango", ((na < 0) | (na > 10)).sum(), len(actas))
hallazgo("actas_docentes.xlsx", "La materia viene por nombre, no por id_materia", len(actas), len(actas))
leg = actas.Legajo.str.upper().where(actas.Legajo.str.upper().str.startswith("L"), "L" + actas.Legajo)
ac = (actas.assign(legajo=leg, nota_acta=na).merge(alumnos[["legajo", "id_alumno"]], on="legajo")
           .merge(materias.rename(columns={"nombre": "Materia"})[["Materia", "id_materia"]], on="Materia")
           .merge(notas_25.astype({"calificacion": float}), left_on=["id_alumno", "id_materia"], right_on=["id_estudiante", "id_materia"]))
hallazgo("actas_docentes.xlsx", "Misma nota que notas_2025-1c.csv (acta × 10 = CSV): son dos fuentes del mismo dato", (ac.nota_acta * 10 == ac.calificacion).sum(), len(ac))

# --- Lo externo: becas ---
hallazgo("becas.csv", "DNI con puntos ('35.527.738')", becas.dni.str.contains(r"\.").sum(), len(becas))
dni_b = becas.dni.str.replace(".", "", regex=False)
hallazgo("becas.csv", "DNI que no existe en la base académica (aun sin puntos)", (~dni_b.isin(alumnos.dni)).sum(), len(becas))
hallazgo("becas.csv", "Becas de personas con dos legajos (el DNI matchea dos filas)", dni_b.isin(pares.dni).sum(), len(becas))

# --- Lo externo: encuesta (XLSX) ---
hallazgo("encuesta_satisfaccion.xlsx", "Puntaje fuera de la escala 1 a 5", (~encuesta.Puntaje.between(1, 5)).sum(), len(encuesta))
hallazgo("encuesta_satisfaccion.xlsx", "Materia mal escrita ('Matería 11')", (~encuesta.Materia.isin(materias.nombre)).sum(), len(encuesta))
hallazgo("encuesta_satisfaccion.xlsx", "Comentario vacío", encuesta.Comentario.isna().sum(), len(encuesta))

hallazgos = pd.DataFrame(hallazgos)
hallazgos


,fuente,problema,casos,de,%
0,alumnos,"Personas con dos legajos (mismo DNI, nombre y ...",60,2000,3.000
1,alumnos,Pares duplicados con estado_final distinto ent...,14,60,23.300
2,alumnos,plan_estudio guarda solo el plan actual (los q...,300,300,100.000
3,cursadas,Cursadas de alumnos que no existen en la base ...,25,15315,0.200
4,cursadas,Legajos duplicados sin ninguna cursada,60,60,100.000
5,cursadas,Estado 'abandonó': aparece SOLO en alumnos con...,547,547,100.000
6,biblioteca.jsonl,Préstamos que identifican al alumno por legajo...,1126,10946,10.300
7,biblioteca.jsonl,El archivo no viene ordenado por fecha (evento...,5481,10946,50.100
8,notas_2025-1c.csv,Cambio de esquema: id_alumno → id_estudiante y...,1,1,100.000
9,notas_2025-1c.csv,Cambio de escala: notas de 0 a 100 en vez de 0...,7316,7334,99.800


### Qué hacemos con cada problema

Cada hallazgo de la tabla de arriba tiene una decisión y un módulo donde se resuelve. Lo que no se puede corregir con una regla clara **no se adivina**: va a cuarentena, con un motivo.

| # | Problema | Qué hacemos | Módulo |
| --- | --- | --- | --- |
| 1 | 60 personas con dos legajos | La persona se identifica por DNI. Se arma una tabla de equivalencias legajo → persona y queda como vigente el legajo con actividad (L2xxxx). Si los dos legajos tienen un `estado_final` distinto (14 casos), vale el del legajo vigente. | 2 y 3 |
| 2 | `plan_estudio` pisa el plan anterior | La historia del plan se reconstruye con `cambios_plan`: una versión con el plan anterior hasta `fecha_cambio` y otra con el plan nuevo desde esa fecha (SCD tipo 2). | 3 |
| 3 | 25 cursadas de alumnos inexistentes | Cuarentena. No entran al hecho porque romperían la integridad referencial. | 4 y 6 |
| 4 | El estado «abandonó» de la cursada equivale al objetivo | No se usa tal cual como variable del modelo. Definimos una fecha de corte y el modelo solo ve lo anterior a ese corte. | 7 |
| 5 | Biblioteca identifica a veces por legajo | En la zona curada se traduce el legajo a `id_alumno` con la tabla de equivalencias. | 4 |
| 6 | Biblioteca desordenada | Se carga por fecha del evento y no por orden de llegada. Para que sea idempotente, la clave es (alumno, ISBN, fecha y hora). | 4 y 5 |
| 7 | Las columnas de notas cambian entre cuatrimestres | Un contrato por cuatrimestre dice cómo se llama cada columna. Si llega un archivo con columnas desconocidas, va a cuarentena entero y se avisa. | 4 y 6 |
| 8 | Escala 0–100 en 2025-1c | Se normaliza a 0–10. La escala de cada archivo queda declarada en el contrato. | 4 |
| 9 | Notas fuera de rango (CSV y actas) | Cuarentena fila por fila y una regla de calidad con umbral. | 6 |
| 10 | Cursada «aprobada» con nota menor a 4 | Se marca como inconsistencia. El estado lo define la base académica y la nota la define el acta. | 6 |
| 11 | Actas: encabezado en la fila 4, cuatrimestre en A2, legajos sin «L» o en minúscula, coma decimal, «AUS» y materia por nombre | Se leen con encabezado fijo y el cuatrimestre sale de la celda A2. El legajo se normaliza (mayúscula y «L»), la coma pasa a punto, «AUS» queda como nota nula con la marca `ausente` y la materia se traduce a `id_materia`. | 4 |
| 12 | Actas y CSV 2025-1c traen la misma nota | El acta es la fuente de verdad (es el documento del docente). El CSV se usa para controlar el acta: si las dos no coinciden, se genera una alerta. | 4 y 6 |
| 13 | Becas: DNI con puntos, 20 sin alumno y 17 de personas duplicadas | Se quitan los puntos y se resuelve a persona. Las becas sin alumno van a cuarentena. | 4 |
| 14 | Encuesta: puntajes fuera de 1–5, «Matería» y comentarios vacíos | Se corrige el nombre de la materia (sin tilde) y los puntajes fuera de escala van a cuarentena. Es anónima: se usa solo agregada por materia y cuatrimestre, nunca cruzada con alumnos. | 4 y 6 |
| 15 | Datos personales: DNI, nombre, fecha de nacimiento, correo y legajo (en alumnos, becas, actas y biblioteca) | Se clasifican, se enmascaran fuera de la zona cruda y se definen accesos por rol. | 9 |

### Lo que el caso no declara: nuestros supuestos

| # | Supuesto | Por qué |
| --- | --- | --- |
| S1 | La persona es el DNI. El legajo vigente es el que tiene cursadas. | Los 60 legajos L3xxxx no tienen ninguna cursada y comparten DNI, nombre y correo con un L2xxxx. |
| S2 | La nota oficial de 2025-1c es la del acta del docente. | Es el documento firmado. El CSV coincide en el 97 %, y el resto son los ausentes. |
| S3 | En 2025-1c la escala es 0–100 y se divide por 10. Se aprueba con 4. | El 99,8 % de las notas de 2025-1c supera 10. Multiplicada por 10, la nota del acta coincide con el CSV. |
| S4 | La encuesta puntúa de 1 a 5. | El 98 % de las respuestas cae entre 1 y 5. Los 0, 6 y 10 son errores de carga. |
| S5 | Las hojas de las actas (Ciencias, Humanidades, Gestión) no indican carrera. | Cada hoja mezcla alumnos de las cinco carreras. |
| S6 | Hay materias compartidas entre carreras. | El 80 % de las cursadas es de una materia asignada a otra carrera. No usamos esa asignación como regla de calidad. |
| S7 | La fecha de inicio del plan anterior es la primera actividad del alumno. | El caso no trae la fecha de alta en el plan. |


## Qué tiene que recorrer tu trabajo

| Módulo | Qué se entrega |
| --- | --- |
| 1 · Arquitectura moderna | Un diagrama con fuentes, almacenamiento, procesamiento y consumo, y la justificación de la elección (warehouse, lake o lakehouse) con **una alternativa descartada**. |
| 2 · Modelado | Modelo conceptual, lógico y físico del dominio y un modelo dimensional: un hecho de cursada y nota con las dimensiones alumno (con el historial de plan), materia y tiempo. Con su DDL. |
| 3 · PL/SQL | Un procedimiento (o un conjunto de sentencias SQL) que cargue la dimensión alumno con historia (SCD tipo 2) y resuelva los alumnos duplicados, con manejo de errores y una medición de tiempos antes y después de optimizarlo. |
| 4 · Ingestión (NiFi) | El flujo para ingerir la base académica, los préstamos de biblioteca, las notas por cuatrimestre, las actas de los docentes (XLSX), las becas y la encuesta de satisfacción (XLSX): procesadores, conexiones, manejo de errores y monitoreo. Se implementa en NiFi o, si no hay entorno, como diagrama más un script equivalente que cargue los datos entregados. |
| 5 · Orquestación (Airflow) | Un DAG con dependencias, calendario (diario durante el cierre de actas y semanal el resto del año), reintentos e idempotencia, y la decisión sobre los datos que llegan tarde. |
| 6 · Governance | Al menos cinco reglas de calidad **medibles**, aplicadas a los datos entregados, con sus resultados; roles (dueño y responsable) y un contrato de datos. |
| 7 · MLOps | Un modelo de riesgo de abandono: ficha de versión, criterios de promoción, monitoreo y marcha atrás, cuidando que no use información posterior al evento. |
| 8 · Plataformas | Las zonas de datos (cruda, curada, variables y consumo) con sus reglas de paso; si hay un modelo, la arquitectura para Machine Learning. |
| 9 · Seguridad y metadatos | Clasificación de los datos (DNI, nombre, fecha de nacimiento, correo y legajo), enmascaramiento y accesos por rol; un catálogo mínimo con el linaje de un indicador. |
| 10 · Proyecto | Una matriz de riesgos, un backlog priorizado y una defensa de 8 minutos. |

**Sobre el modelo (central):** Un modelo de riesgo de abandono: ficha de versión, criterios de promoción, monitoreo y marcha atrás, cuidando que no use información posterior al evento.

## 1 · Arquitectura moderna

Escribí la arquitectura (fuentes propias y externas, almacenamiento, procesamiento, consumo) y la justificación, con **una alternativa descartada**. Podés usar un diagrama de texto o Mermaid.

| Campo | Contenido |
| --- | --- |
| Decisión |  |
| Restricción comprobable |  |
| Opción elegida y costo que se acepta |  |
| **Alternativa descartada y por qué** |  |
| Control que detecta el fallo y qué se hace cuando suena |  |

## 2 · Modelado

Modelo conceptual, lógico y físico, y un modelo dimensional: un hecho de cursada y nota con las dimensiones alumno (con el historial de plan), materia y tiempo. Escribí el DDL y creá las tablas en tu almacén.

In [6]:
almacen = dwh()
almacen.executescript("""
-- Tu DDL
""")

## 3 · PL/SQL

Un procedimiento que cargue la dimensión alumno con historia (SCD tipo 2) y resuelva los alumnos duplicados. SQLite no tiene PL/SQL: escribilo como sentencias SQL ejecutadas desde una función de Python con manejo de errores, y medí los tiempos antes y después de optimizarlo con `medir(...)`.

In [7]:
def cargar():
    # Tu carga: SQL + manejo de errores
    pass

# _, segundos = medir(cargar)

## 4 · Ingestión

El flujo para ingerir la base académica, los préstamos de biblioteca, las notas por cuatrimestre, las actas de los docentes (XLSX), las becas y la encuesta de satisfacción (XLSX). Si tenés Apache NiFi, diseñalo ahí; si no, escribí el diagrama (procesadores, conexiones, manejo de errores y monitoreo) y un script equivalente. Cada archivo externo es un origen distinto, con su propio formato y sus propios problemas.

In [8]:
# Tu ingestión
externos()

actas_docentes.xlsx                            44 KB  hojas: Ciencias, Humanidades, Gestión
becas.csv                                      10 KB
encuesta_satisfaccion.xlsx                     20 KB  hojas: Respuestas
notas_2024-2c.csv                             187 KB
notas_2025-1c.csv                             179 KB


## 5 · Orquestación

Un DAG con dependencias, calendario (diario durante el cierre de actas y semanal el resto del año), reintentos e idempotencia, y la decisión sobre los datos que llegan tarde. Con `correr_dag(tareas, dependencias)` lo ejecutás en este notebook.

In [9]:
def extraer(): pass
def transformar(): pass
def cargar_dwh(): pass

# estados = correr_dag({"extraer": extraer, "transformar": transformar, "cargar": cargar_dwh},
#                      {"transformar": ["extraer"], "cargar": ["transformar"]})

## 6 · Governance

Al menos **cinco reglas de calidad medibles** aplicadas a los datos propios y externos, con sus resultados, más los roles (dueño y responsable) y un contrato de datos. «No hay nulos» no alcanza como definición de calidad.

In [10]:
# regla("nombre", valor_medido, umbral)

In [11]:
pd.DataFrame(resultados_calidad)

""


## 7 · MLOps

Un modelo de riesgo de abandono: ficha de versión, criterios de promoción, monitoreo y marcha atrás, cuidando que no use información posterior al evento.

In [12]:
# Ficha de versión, criterios de promoción, monitoreo y marcha atrás (si corresponde)

## 8 · Plataformas

Las zonas de datos (cruda, curada, variables y consumo) con sus reglas de paso. Si hay un modelo, la arquitectura para Machine Learning.

In [13]:
zonas()

['consumo', 'cruda', 'curada', 'variables']

## 9 · Seguridad y metadatos

Clasificación de los datos (DNI, nombre, fecha de nacimiento, correo y legajo), enmascaramiento y accesos por rol, y un catálogo mínimo con el linaje de un indicador.

In [14]:
clasificacion = {
    # "tabla.columna": "público | interno | confidencial | sensible",
}
catalogo = {
    # "indicador": {"dueño": "", "responsable": "", "definición": "", "frescura": "", "linaje": []},
}

## 10 · Proyecto: riesgos, backlog y defensa

**Matriz de riesgos**

| Riesgo | Probabilidad | Impacto | Cómo se detecta | Cómo se contiene y se recupera | Responsable |
| --- | --- | --- | --- | --- | --- |
|  |  |  |  |  |  |

**Backlog priorizado**

| Prioridad | Qué | Por qué ahora | Esfuerzo | Depende de |
| --- | --- | --- | --- | --- |
|  |  |  |  |  |

**Guion de la defensa de 8 minutos:** problema y decisión (1) · diseño (3) · evidencia (2) · fallas y recuperación (1) · lo que queda abierto (1).

### Antes de entregar

- [ ] Reinicié el kernel y corrí todo el notebook de punta a punta.
- [ ] Escribí qué suponemos y el caso no declara.
- [ ] Elegimos la herramienta después de fijar el requisito.
- [ ] Cada decisión tiene una alternativa descartada y el costo que aceptamos.
- [ ] Encontramos los problemas de las fuentes propias y externas y dijimos qué hacemos con cada uno.
- [ ] Cada control dice **qué se hace cuando falla** y quién decide.
- [ ] La carga es idempotente y se puede reprocesar un día cualquiera.
- [ ] Los datos sensibles están clasificados, enmascarados y con acceso por rol.
- [ ] Si el caso lleva un modelo, no usa información posterior al evento.
- [ ] Ensayamos la defensa en 8 minutos.